In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import ScalarFormatter, AutoLocator
from matplotlib.axes import Axes
import seaborn as sns
from seaborn import (axes_style, plotting_context)
import sklearn.metrics as skm
import colorcet as cc
import sklearn as sk
import sklearn.decomposition as decomp
import sklearn.pipeline as pipe
import sklearn.neighbors as nbr
import sklearn.base as skbase
import sklearn.model_selection as skms
import sklearn.preprocessing as skpr
import sklearn.linear_model as sklm
import pickle
import os
import joblib
import itertools
import statsmodels.api as sma
import statsmodels as sm
import statsmodels.formula.api as sf
from statsmodels.discrete.truncated_model import (
    HurdleCountModel,
    TruncatedLFNegativeBinomialP
)
import glob
import anndata as ann
from typing import Literal, Sequence
import scipy as sp
import pymc as pm
import arviz as az
from pytensor.tensor import TensorVariable
import pytensor.tensor as pt
import cvxpy as cx
import pygam as pg


In [ ]:
merged_data = sc.read_h5ad("data/merged_w_others_filt.h5ad")
merged_data

In [ ]:
merged_data.obs["idents"] = merged_data.obs["idents"].str.replace(" ", "_")
df = pd.get_dummies(merged_data.obs[["idents"]])
df = df.loc[df.any(axis= 1)]
df = df.astype(int)
symbol = "FTL"
gene = merged_data.var.query("gene_symbol == @symbol").index[0]
df["counts"] = merged_data[
    :,
    gene
].to_df().iloc[:, 0]
df["sum"] = merged_data.obs["sum"]

In [ ]:
symbol = "FTL"
gene = merged_data.var.query("gene_symbol == @symbol").index[0]
data = merged_data[
    :,
    gene
].to_df("norm").iloc[:, 0]
ax = sns.histplot(
    data= data[merged_data.obs.query("idents == 'Ribosomal_Ewing'").index],
)
ax.axvline(x= data.mean())
ax.set_yscale("log")

In [ ]:
data = merged_data[
    :,
    gene
].to_df()
data.columns = ["counts"]
data["exposure"] = merged_data.obs["sum"] / merged_data.obs["sum"].median()
data["idents"] = merged_data.obs["idents"]
data = data.eval("norm_counts = counts / exposure")
ax = sns.histplot(
    data= data.loc[merged_data.obs.query("idents == 'Ribosomal_Ewing'").index, "norm_counts"],
)
ax.set_yscale("log")

In [ ]:
data["idents"] = data["idents"].astype("category")

In [ ]:
fit_df = data.copy()
fit_df["norm_counts"] = fit_df["norm_counts"].where(fit_df["norm_counts"] > 0, np.nan)
m = sf.glm(
    "counts ~ C(idents) + np.log(exposure) + 0",
    data= fit_df,
    family= sma.families.Gamma(link= sma.families.links.Log()),
    missing= "drop",
)
res = m.fit(
)
res.summary()

In [ ]:
data.to_csv("data/FTL_test.csv")

In [ ]:
res.params

In [ ]:
column_name = "Ribosomal_Ewing"
exp_factor = sp.stats.mstats.gmean(
    data.query(f"idents == '{column_name}'")["exposure"]
)
mu = np.exp(2.066 + 0.315 * np.log(exp_factor))
alpha = 1 / np.exp(-0.7138) 
beta = mu / alpha

display(column_name)
hist_data = data.query(f"idents == '{column_name}'")[["counts"]]
hist_data = hist_data.query("counts > 0")
est = sp.stats.gamma.rvs(a= alpha, scale= beta, size= hist_data.shape[0])
hist_data["est"] = est
ax = sns.ecdfplot(
    hist_data,
)
# ax.set_yscale("log")

In [ ]:
fit_df = data.copy()
fit_df["norm_counts"] = fit_df["norm_counts"].where(fit_df["norm_counts"] > 0, np.nan)
m = sf.negativebinomial(
    "counts ~ C(idents) + 0",
    data= fit_df,
    exposure= data["exposure"],
    missing= "drop",
)
res = m.fit(
    method= "minimize",
    min_method= "trust-krylov",
    disp= False,
)
res.summary()

In [ ]:
column_name = "Transition_Ribo_Ewing"
exp_factor = sp.stats.mstats.gmean(
    data.query(f"idents == '{column_name}'")["exposure"]
)
mu = np.exp(res.params[f"C(idents)[{column_name}]"] + np.log(exp_factor))
a = res.params["alpha"]
var = mu + a * mu ** 2
p = mu / var
n = mu ** 2 / (var - mu)

display(column_name)
hist_data = data.query(f"idents == '{column_name}'")[["counts"]] 
est = sp.stats.nbinom.rvs(n, p, size= hist_data.shape[0])
hist_data["est"] = est
ax = sns.histplot(
    hist_data,
)
ax.set_yscale("log")

In [ ]:
m = HurdleCountModel.from_formula(
    "counts ~ C(idents) + np.log(exposure)",
    data= data,
    dist= "negbin",
)
res = m.fit(
    method= "minimize",
    min_method= "trust-ncg",
    max_trust_radius= 10,
    # disp= False,
)
res.summary()

In [ ]:
column_number = 5
mu = np.exp(res.params[f"x{column_number}"]) + np.exp(res.params[f"x7"])
a = res.params["alpha"]
var = mu + a * mu ** 2
p = mu / var
n = mu ** 2 / (var - mu)

# alpha = mu ** 2 / var
# beta = mu / var
column_name = df.columns[column_number - 1]
est = sp.stats.nbinom.rvs(n, p, size= df[column_name].sum())
# est = sp.stats.gamma.rvs(a= alpha, scale= (1/beta), size= df[column_name].sum())
display(column_name)
hist_data = df.query(f"{column_name} == 1")[["counts"]]
hist_data["est"] = est
sns.histplot(
    hist_data,
).set_yscale("log")

In [ ]:
merged_data.var.sort_values("mean").sample(10)

In [ ]:
ident = "Ribosomal_Ewing"
symbol = "FTL"
gene = merged_data.var.query("gene_symbol == @symbol").index[0]
data = merged_data[
    :,
    gene
].to_df("norm").iloc[:, 0]
obs_data = data[merged_data.obs.query("idents == @ident").index].copy()
nonz_data = obs_data[obs_data > 0]
gamma_params = sp.stats.gamma.fit(
    nonz_data,
    floc = 0,
)
dist = sp.stats.gamma(*gamma_params)
ls = np.linspace(0.01, gamma_params[0] * 2, 200)
i = np.argmax(dist.pdf(ls))
cutoff = ls[i]

scaled_data = obs_data.copy().where(obs_data > cutoff, cutoff)
cdata = sp.stats.CensoredData(
    scaled_data[scaled_data > cutoff],
    left= scaled_data[scaled_data == cutoff],
    )
fit_gamma = sp.stats.gamma.fit(
    cdata,
    floc = 0,
)
est_data = sp.stats.gamma.rvs(*fit_gamma, size= obs_data.shape[0])
est_data.sort()
df = obs_data.rename("obs").sort_values().to_frame()
df["scaled"] = scaled_data
df["est"] = est_data
clip_loc = df["est"].quantile((df["scaled"] == 0).mean())
df = df.eval("est_clip = est.where(est > @cutoff, @cutoff)")

ax = sns.histplot(
    obs_data,
)
ax.set_yscale("log")
ax.axvline(cutoff, color = "r")

In [ ]:
np.log(fit_gamma[0])

In [ ]:
fit_gamma

In [ ]:
sns.ecdfplot(
    df[["obs", "scaled", "est", "est_clip"]],
    alpha= 0.65
)

In [ ]:
ax = sns.histplot(
    df,
    element= "step",
    alpha= 0.1
)
# ax.set_yscale("log")
# ax.set_ylim(top= 150)

In [ ]:
lower = df["scaled"].min()

    
with pm.Model():
    mu_ = pm.Normal("mu", mu= 1, sigma= 10)
    sigma_ = pm.HalfNormal("sigma", sigma= 10)

    pm.Censored(
        "rna",
        pm.Gamma.dist(
            mu= mu_,
            sigma= sigma_,
        ),
        lower= lower,
        observed= df["scaled"],
    )
    idata = pm.sample(nuts_sampler= "numpyro", tune= 3000, draws= 1000)
    thinned_idat = idata.sel(draw= slice(None, None, 50))
    pm.sample_posterior_predictive(thinned_idat, extend_inferencedata= True)



In [ ]:
az.plot_ppc_dist(thinned_idat)

In [ ]:
az.summary(idata)

In [ ]:
m = idata.posterior["mu"].mean()
v = idata.posterior["sigma"].mean() ** 2
#Gamma
a = m ** 2 / v
b = m / v
est_data = sp.stats.gamma.rvs(
    a= a,
    scale = 1 / b,
    size= obs_data.shape[0],
)
hist_data = pd.Series(obs_data).rename("obs").to_frame().copy()

hist_data["est"] = est_data
hist_data["est_clip"] = est_data.copy()
hist_data.loc[hist_data["est_clip"] < 0, "est_clip"] = 0
hist_data = hist_data.melt()
ax = sns.ecdfplot(
    hist_data,
    x= "value",
    hue= "variable",
    alpha= 0.3,
)
# ax.set_xlim(left= -2, right= 5)

In [ ]:
ax = sns.kdeplot(
    hist_data,
    x= "value",
    hue= "variable",
    alpha= 0.6
)
# ax.set_xlim(right= 10)
# ax.set_yscale("symlog")

In [ ]:
ax = sns.histplot(
    hist_data,
    x= "value",
    hue= "variable",
)
ax.set_yscale("log")
# ax.set_xlim(right= 2)
# ax.set_ylim(top= 250)

In [ ]:
merged_data.var.sort_values("mean") #.sample(10)

In [ ]:
df = merged_data.obs[["idents"]]
symbol = "FTL"
gene = merged_data.var.query("gene_symbol == @symbol").index[0]
df["counts"] = merged_data[
    :,
    gene
].to_df().iloc[:, 0]
df["sum"] = merged_data.obs["sum"]
means = df.groupby("idents")["counts"].sum()
totals = df.groupby("idents")["sum"].sum()
df = df.query("counts > 0")
# df["counts"] += 1
df = df.eval("distance = (sum) / counts")
df

In [ ]:
m = sf.glm(
    "sum ~ C(idents) + np.log(counts):C(idents) + 0",
    data= df,
    family= sma.families.Gamma(link= sma.families.links.Log()),
)
res = m.fit()
res.summary()

In [ ]:
column_name = "Immune"
param_name = f"C(idents)[{column_name}]"
meanl_counts = np.log(df.query(f"idents == '{column_name}'")["counts"].mean())

mu = np.exp(res.params[param_name] + res.params[f"np.log(counts):{param_name}"] * meanl_counts)
alpha = 1 / m.scale 
beta = mu / alpha


hist_data = df.query(f"idents == '{column_name}'")[["sum"]]  #.eval("scaled = counts / expose")[["scaled"]]
est = sp.stats.gamma.rvs(a= alpha, scale= beta, size= hist_data.shape[0])
hist_data["est"] = est
ax = sns.histplot(
    hist_data,
)
# ax.set_yscale("log")
ax.axvline((mu - mu / alpha))

In [ ]:
totals[column_name] / mu

In [ ]:
means

In [ ]:
lam = cx.Variable()
shift = cx.Parameter(value= 1, nonneg= True)
constraints = [
    lam >= 0.1,
    lam <= 100,
    shift >= 0,
    shift <= 10,
]
x = data.values.copy()
x = x[x != 0]
psi = sp.special.digamma(x + 1)
log_gamma = sp.special.gammaln(x + 1)
censor_loc = data.mean()
k = np.arange(censor_loc, data.max())
log_gamma_k = sp.special.gammaln(k + 1)
tests = 100
res = np.zeros(tests)
lams = np.zeros(tests)
objective = cx.Maximize(cx.sum((x + shift) * cx.log(lam) - lam - cx.loggamma((x + shift) + 1))) # + cx.sum(cx.log(cx.cumsum(1 - cx.exp(k * cx.log(lam) - cx.loggamma(k) - lam)))))
prob = cx.Problem(objective, constraints)
res = prob.solve()

In [ ]:
l = lam.value
shift = 0.5
# m = multiplyer[res == res.max()][0]
hist_data = pd.Series(x).rename("obs").to_frame().copy()
x = np.linspace(0.1, data.max(), 1000)
est_data = np.exp((x) * np.log(l) - l - sp.special.gammaln((x + 1)))
ax = sns.lineplot(
    x= x,
    y= est_data,
    color= "red",
)
sns.kdeplot(
    hist_data,
    ax= ax,
)
ax.set_ylim(top= 2.5)

In [ ]:
def log_like(lam, x):
    return cx.log(cx.log(lam) - sp.special.digamma(x + 1)) - cx.loggamma(x + 1) + x * cx.log(lam) - lam

